In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# -----------------------------------------------------------------------------
# 1. HYPERPARAMETERS
# -----------------------------------------------------------------------------
batch_size = 16       
block_size = 32       
max_iters = 1000      
eval_interval = 100   
learning_rate = 1e-3
n_embd = 64           
n_head = 4            
n_layer = 4           
dropout = 0.0

# SIMPLIFIED: Explicit if/else instead of inline conditional
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

print(f"Running on device: {device}")

# -----------------------------------------------------------------------------
# 2. DATASET & TOKENIZER
# -----------------------------------------------------------------------------
text = """
The quick brown fox jumps over the lazy dog.
Artificial intelligence is transforming the world.
To be or not to be, that is the question.
I am building a neural network from scratch using PyTorch.
""" * 10 

chars = sorted(list(set(text)))
vocab_size = len(chars)

# SIMPLIFIED: Replaced dictionary comprehensions with standard for loops
stoi = {}
for i, ch in enumerate(chars):
    stoi[ch] = i

itos = {}
for i, ch in enumerate(chars):
    itos[i] = ch

# SIMPLIFIED: Replaced lambda functions with explicit functions
def encode(input_string):
    encoded_list = []
    for character in input_string:
        encoded_list.append(stoi[character])
    return encoded_list

def decode(input_list):
    decoded_string = ""
    for integer in input_list:
        decoded_string = decoded_string + itos[integer]
    return decoded_string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    # SIMPLIFIED: Explicit if/else
    if split == 'train':
        data_source = train_data
    else:
        data_source = val_data
        
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    
    # SIMPLIFIED: Replaced list comprehensions with standard for loops
    x_list = []
    y_list = []
    for i in ix:
        x_list.append(data_source[i : i + block_size])
        y_list.append(data_source[i + 1 : i + block_size + 1])
        
    x = torch.stack(x_list)
    y = torch.stack(y_list)
    
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    
    for split in ['train', 'val']:
        losses = torch.zeros(10)
        for k in range(10):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
        
    model.train()
    return out

# -----------------------------------------------------------------------------
# 3. ARCHITECTURE
# -----------------------------------------------------------------------------

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__() 
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   
        q = self.query(x) 
        
        wei = q @ k.transpose(-2, -1) * (C ** -0.5) 
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) 
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__() 
        
        # SIMPLIFIED: Removed list comprehension
        head_list = []
        for _ in range(num_heads):
            head_list.append(Head(head_size))
            
        self.heads = nn.ModuleList(head_list)
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # SIMPLIFIED: Removed list comprehension
        head_outputs = []
        for h in self.heads:
            head_outputs.append(h(x))
            
        out = torch.cat(head_outputs, dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__() 
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__() 
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))   
        x = x + self.ffwd(self.ln2(x)) 
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__() 
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        
        # SIMPLIFIED: Removed list comprehension
        block_list = []
        for _ in range(n_layer):
            block_list.append(Block(n_embd, n_head))
            
        self.blocks = nn.Sequential(*block_list)
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) 
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) 
        x = tok_emb + pos_emb 
        x = self.blocks(x)    
        x = self.ln_f(x)      
        logits = self.lm_head(x) 

        # SIMPLIFIED: Clearer target check
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] 
            probs = F.softmax(logits, dim=-1) 
            idx_next = torch.multinomial(probs, num_samples=1) 
            idx = torch.cat((idx, idx_next), dim=1) 
        return idx

# -----------------------------------------------------------------------------
# 4. TRAINING LOOP & INFERENCE
# -----------------------------------------------------------------------------
model = GPTLanguageModel(vocab_size)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Starting training...")
for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss(model)
        print(f"Step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("\n--- Training Complete. Generating Text ---\n")

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text = decode(model.generate(context, max_new_tokens=200)[0].tolist())
print(generated_text)

Running on device: cuda
Starting training...
Step 0: train loss 3.6382, val loss 3.6385
Step 100: train loss 1.1791, val loss 1.1907
Step 200: train loss 0.2194, val loss 0.2041
Step 300: train loss 0.1112, val loss 0.1132
Step 400: train loss 0.0987, val loss 0.0997
Step 500: train loss 0.0926, val loss 0.0927
Step 600: train loss 0.0997, val loss 0.1043
Step 700: train loss 0.0873, val loss 0.0864
Step 800: train loss 0.0831, val loss 0.0800
Step 900: train loss 0.0818, val loss 0.0808

--- Training Complete. Generating Text ---


The quick brown fox jumps over the lazy dog.
Artificial intelligence is transforming the world.
To be or not to be, that is the question.
I quiam building a neural network from scratch using PyTorch.

